### Configure Path for src imports

In [1]:
import sys 
import os 
sys.path.append(os.path.abspath(".."))

### Importing Libraries

In [11]:
from datasets import load_dataset 
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [13]:
from src.ml_utils import load_ml_data, build_pipeline, train_and_evaluate, get_tfidf, save_model

### Load Dataset

In [4]:
X_train, X_test, y_train, y_test = load_ml_data()

### Training

In [9]:
models = {
    "logistic_regression": LogisticRegression(),
    "linear_svm": LinearSVC(),
    "naive_bayes": MultinomialNB(),
    "random_forest": RandomForestClassifier(),
    "knn": KNeighborsClassifier()
}

results = {}
trained_pipelines = {}

for name, model in models.items():
    pipeline = build_pipeline(model)

    acc, trained_pipeline = train_and_evaluate(
        pipeline, X_train, y_train, X_test, y_test
    )

    results[name] = acc
    trained_pipelines[name] = trained_pipeline

    print(f"{name}: {acc:.4f}")

logistic_regression: 0.8892
linear_svm: 0.8938
naive_bayes: 0.8768
random_forest: 0.8592
knn: 0.7856


### Logistic Regression

In [12]:
pipeline = Pipeline([
    ("tfidf", get_tfidf()),
    ("clf", LogisticRegression(max_iter=1000))
])

param_grid = {
    "clf__C": [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
preds = best_model.predict(X_test)
acc = accuracy_score(y_test, preds)

print("Accuracy:", acc)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Accuracy: 0.8964


In [14]:
save_model(best_model, "../models/ml/logistic_regression.joblib")

### SVM

In [15]:
pipeline = Pipeline([
    ("tfidf", get_tfidf()),
    ("clf", LinearSVC())
])

param_grid = {
    "clf__C": [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
preds = best_model.predict(X_test)
acc = accuracy_score(y_test, preds)

print("Accuracy:", acc)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Accuracy: 0.8938


In [16]:
save_model(best_model, "../models/ml/linear_svc.joblib")

### Naive Bayes

In [17]:
pipeline = Pipeline([
    ("tfidf", get_tfidf()),
    ("clf", MultinomialNB())
])

param_grid = {
    "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
preds = best_model.predict(X_test)
acc = accuracy_score(y_test, preds)

print("Accuracy:", acc)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Accuracy: 0.8768


In [18]:
save_model(best_model, "../models/ml/multinomial_nb.joblib")